# Chapter 1 &mdash; The Chomsky Hierarchy: Machines, Patterns and Grammars

**Concept 17 of the Chapter 1 decomposition:** *Chomsky's Grammars and the Chomsky Hierarchy*

Chomsky classified <b>grammars</b> from an unrelated direction &mdash; and they landed exactly on the four machine classes.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Chomsky-Hierarchy/Concept-Chomsky-Hierarchy.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Chomsky stratified grammars into **Type-0, 1, 2, 3**. They correspond exactly to the
patterns recognised by TM, LBA, PDA and FA.

| Type | Grammar restriction | Machine | Languages |
|---|---|---|---|
| 3 | purely left- or right-linear | DFA/NFA | regular |
| 2 | one nonterminal on the LHS | PDA | context-free |
| 1 | $\mid$LHS$\mid \le \mid$RHS$\mid$ | LBA | context-sensitive |
| 0 | no restriction | TM | recursively enumerable |

Machines, patterns and grammars are **three facets of one thing** &mdash; a second
convergence, after Concept 6.

## 2. Definitions

### A Type-3 (right-linear) grammar

Every rule is `A -> a B` or `A -> ''`. That shape **is** a DFA in disguise.

In [ ]:
type3 = {
    'S': [('0', 'S'), ('1', 'F')],
    'F': [('0', 'F'), ('1', 'S'), ('', None)],   # ('',None) = accept here
}

def derive_type3(g, start, s):
    """Run a right-linear grammar like a DFA."""
    nt = start
    for ch in s:
        nxt = [B for (a, B) in g[nt] if a == ch and B is not None]
        if not nxt:
            return False
        nt = nxt[0]
    return any(a == '' for (a, B) in g[nt])

### A Type-2 (context-free) grammar

One nonterminal on the left; the right side may be anything. Nonterminals in the
**middle** are what buys nesting.

In [ ]:
# S -> '' | ( S ) | S S      -- the Dyck language
def derive_type2(s):
    """Recognise the Dyck language, the language of the grammar above."""
    d = 0
    for ch in s:
        d += 1 if ch == '(' else -1
        if d < 0: return False
    return d == 0

<!-- nav-strip -->

---

&larr;&nbsp;[Ch1&nbsp;16.&nbsp;Machine Classes as Programming Restrictions](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Machines-As-Programming-Restrictions/Concept-Machines-As-Programming-Restrictions.ipynb) &nbsp;&middot;&nbsp; [**Chapter 1** index](https://github.com/ganeshutah/Jove/blob/master/Chapter1/README.md) &nbsp;&middot;&nbsp; [Ch1&nbsp;18.&nbsp;Why This Material Matters for Lifelong Learning](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Why-This-Matters/Concept-Why-This-Matters.ipynb)&nbsp;&rarr;

---

## 3. Tests

The Type-3 grammar accepts exactly "an odd number of `1`s" &mdash; a regular language.

In [ ]:
for s in ['', '1', '11', '101', '0110', '111']:
    print("%-6s odd # of 1s? %-6s  grammar says %s"
          % (repr(s), s.count('1') % 2 == 1, derive_type3(type3, 'S', s)))
assert all(derive_type3(type3, "S", s) == (s.count("1") % 2 == 1)
           for s in ["", "1", "11", "101", "0110", "111"])

The Type-2 grammar accepts nesting, which no Type-3 grammar can.

In [ ]:
for s in ['', '()', '(())', '()()', '(()', ')(']:
    print("%-8s in Dyck? %s" % (repr(s), derive_type2(s)))
assert derive_type2("(())") and not derive_type2(")(")

The whole hierarchy, with a witness language for each strict inclusion.

In [ ]:
rows = [("Type-3 regular          ", "(01)*         ", "DFA / NFA"),
        ("Type-2 context-free     ", "a^n b^n       ", "PDA"),
        ("Type-1 context-sensitive", "a^n b^n c^n   ", "LBA"),
        ("Type-0 recursively enum.", "{(M,w): M halts on w}", "TM")]
for t, lang, m in rows:
    print("%s  e.g. %-22s  machine: %s" % (t, lang, m))
print()
print("Each class STRICTLY contains the one below it.")
print("Turing machines: 1936.  Finite automata: 1957.  The general case came FIRST.")

## 4. Exercises


1. Write a right-linear grammar for "ends in `01`" and run it with `derive_type3`.
2. `S -> '' | (S) | SS` has `S` in the **middle** of a rule. Which type does that make
   it, and why can a right-linear grammar never do the same?
3. Nondeterminism adds no power for FA, **real** power for PDA, and no power again for
   TM. Where would you place LBA, and why is that still open?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter1/Concept-Chomsky-Hierarchy')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')